# DisasterLens: focused cross-disaster damage assessment

This is a separate Kaggle/T4 entry point. It leaves `notebooks/train.ipynb` and the earlier M1--M3 workflow untouched. It uses only the attached official BRIGHT dataset and the persistent M1-cache dataset.

Each Push & Run performs one deliberately bounded experiment (default: E1 early-fusion U-Net on the standard split for 30 epochs), with live batch and epoch logs. Change `FOCUSED_MODEL`, `FOCUSED_SPLIT`, or `FOCUSED_EPOCHS` in the configuration cell for a subsequent run.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import time

import torch

if not torch.cuda.is_available():
    raise RuntimeError('A Kaggle GPU is required. In Notebook options, select GPU accelerator: Tesla T4.')
gpu_name = torch.cuda.get_device_name(0)
print(f'[Kaggle] GPU: {gpu_name}', flush=True)
if 'T4' not in gpu_name.upper():
    raise RuntimeError(f'This benchmark is configured for a Kaggle Tesla T4; got {gpu_name!r}.')

working = Path('/kaggle/working')
repo_dir = working / 'disaster-lens'
if not (repo_dir / 'cross_disaster_damage_assessment' / 'run.py').is_file():
    if repo_dir.exists():
        raise RuntimeError(f'{repo_dir} exists but is not the required public repository checkout.')
    print('[Kaggle] cloning the public repository', flush=True)
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/kushc2004/disaster-lens.git', str(repo_dir)], check=True)
else:
    print(f'[Kaggle] using repository checkout: {repo_dir}', flush=True)

def has_tiles(path, suffix):
    return path.is_dir() and any(path.glob(f'*{suffix}'))

def nested_modality(root, modality, suffix):
    for candidate in (root / modality, root / modality / modality):
        if has_tiles(candidate, suffix):
            return candidate
    return None

def bright_modalities(root):
    return {
        'pre-event': nested_modality(root, 'pre-event', '_pre_disaster.tif'),
        'post-event': nested_modality(root, 'post-event', '_post_disaster.tif'),
        'target': nested_modality(root, 'target', '_building_damage.tif'),
    }

def bright_roots(input_base):
    if not input_base.is_dir():
        return [], []
    mounts = sorted(path for path in input_base.iterdir() if path.is_dir())
    roots = [input_base / 'bright-dataset', *mounts]
    frontier = mounts
    # Kaggle Studio may namespace an input as datasets/<owner>/<dataset>.
    # Inspect only those two directory levels; never recurse through image tiles.
    for _ in range(2):
        children = []
        for directory in frontier:
            try:
                children.extend(path for path in directory.iterdir() if path.is_dir())
            except OSError:
                pass
        roots.extend(children)
        frontier = children
    roots = list(dict.fromkeys(roots))
    return mounts, [(root, bright_modalities(root)) for root in roots if root.is_dir()]

input_base = Path('/kaggle/input')
mount_deadline = time.monotonic() + 180
while True:
    mounts, inspected = bright_roots(input_base)
    matches = [(root, modalities) for root, modalities in inspected if all(path is not None for path in modalities.values())]
    if len(matches) == 1:
        input_root, modalities = matches[0]
        break
    if len(matches) > 1:
        raise RuntimeError(f'Ambiguous BRIGHT inputs; validated roots: {[str(root) for root, _ in matches]}')
    if time.monotonic() >= mount_deadline:
        raise RuntimeError(f'Official BRIGHT layout was not found after 180 seconds. Visible mounts: {mounts}; inspected roots: {[str(root) for root, _ in inspected]}')
    print(f'[Kaggle] waiting for official BRIGHT layout; mounts: {mounts}; inspected roots: {[str(root) for root, _ in inspected]}', flush=True)
    time.sleep(10)
print(f'[Kaggle] official BRIGHT input root: {input_root}', flush=True)
normalized = working / 'bright-normalized'
if normalized.exists():
    expected = {'pre-event': '_pre_disaster.tif', 'post-event': '_post_disaster.tif', 'target': '_building_damage.tif'}
    if not all(has_tiles(normalized / name, suffix) for name, suffix in expected.items()):
        raise RuntimeError(f'Stale normalized directory at {normalized}; remove it in a new Kaggle session and rerun.')
else:
    normalized.mkdir(parents=True)
    for name, source in modalities.items():
        (normalized / name).symlink_to(source, target_is_directory=True)
dataset_root = normalized.resolve()
os.environ['DISASTERLENS_BRIGHT_ROOT'] = str(dataset_root)
print(f'[Kaggle] official BRIGHT root: {dataset_root}', flush=True)

cache_deadline = time.monotonic() + 180
cache_markers = sorted(input_base.rglob('m1_cache.json')) if input_base.is_dir() else []
while len(cache_markers) != 1:
    if time.monotonic() >= cache_deadline:
        raise RuntimeError(f'Expected exactly one persistent M1-cache marker after 180 seconds; found: {cache_markers}')
    print(f'[Kaggle] waiting for M1-cache input mount; found markers: {cache_markers}', flush=True)
    time.sleep(10)
    cache_markers = sorted(input_base.rglob('m1_cache.json')) if input_base.is_dir() else []
cache_root = cache_markers[0].parent
print(f'[M1 cache] restoring manifest and normalization statistics from {cache_root}', flush=True)
subprocess.run([sys.executable, '-u', 'scripts/restore_kaggle_m1_cache.py', '--cache-root', str(cache_root), '--dataset-root', str(dataset_root), '--repo-root', str(repo_dir)], cwd=repo_dir, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(repo_dir)], check=True)
print(f'[Kaggle] repository: {repo_dir}', flush=True)

In [ ]:
# One Push & Run trains exactly one model/split combination.
# Valid models: 'unet' (E1), 'siamese_resnet18' (E2).
# Valid split names: 'standard', or an event id shown by the prepare step.
FOCUSED_MODEL = 'unet'
FOCUSED_SPLIT = 'standard'
FOCUSED_EPOCHS = 30
BATCH_SIZE = 4
WORKERS = 2

if FOCUSED_MODEL not in {'unet', 'siamese_resnet18'}:
    raise ValueError('FOCUSED_MODEL must be unet or siamese_resnet18')
if FOCUSED_EPOCHS < 1:
    raise ValueError('FOCUSED_EPOCHS must be positive')
print(f'[plan] model={FOCUSED_MODEL}, split={FOCUSED_SPLIT}, epochs={FOCUSED_EPOCHS}, batch_size={BATCH_SIZE}', flush=True)

In [ ]:
def run_step(label, command):
    print('\n' + '=' * 72, flush=True)
    print(label, flush=True)
    print('=' * 72, flush=True)
    print('[command] ' + ' '.join(map(str, command)), flush=True)
    subprocess.run(command, cwd=repo_dir, check=True)

runner = [sys.executable, '-u', 'cross_disaster_damage_assessment/run.py']
run_step('[1/5] Create focused audit and immutable splits from the cached official M1 manifest', runner + ['prepare'])

output_root = repo_dir / 'outputs' / 'cross_disaster_damage_assessment'
split_path = output_root / 'splits' / 'standard.json' if FOCUSED_SPLIT == 'standard' else output_root / 'splits' / 'event_holdout' / f'{FOCUSED_SPLIT}.json'
if not split_path.is_file():
    events_csv = output_root / 'audit' / 'events.csv'
    raise FileNotFoundError(f'Unknown FOCUSED_SPLIT={FOCUSED_SPLIT!r}. Read {events_csv} in the Kaggle output and use an event_id from it.')
run_dir = output_root / 'runs' / FOCUSED_MODEL / split_path.stem
run_step('[2/5] Train with live batch and epoch progress', runner + ['train', '--model', FOCUSED_MODEL, '--split', str(split_path), '--epochs', str(FOCUSED_EPOCHS), '--batch-size', str(BATCH_SIZE), '--workers', str(WORKERS)])
run_step('[3/5] Evaluate validation partition by event and class', runner + ['evaluate', '--run-dir', str(run_dir), '--partition', 'val', '--batch-size', str(BATCH_SIZE), '--workers', str(WORKERS)])
run_step('[4/5] Evaluate test partition by event and class', runner + ['evaluate', '--run-dir', str(run_dir), '--partition', 'test', '--batch-size', str(BATCH_SIZE), '--workers', str(WORKERS)])
run_step('[5/5] Fit validation-only temperature scaling and compile report', runner + ['calibrate', '--run-dir', str(run_dir)])
run_step('[report] Compile all completed focused-run artifacts', runner + ['report'])
print(f'\n[complete] Kaggle outputs are under {output_root}', flush=True)

## Saved artifacts

Kaggle downloads `outputs/cross_disaster_damage_assessment/`. The selected run contains its immutable split, resolved configuration, checkpoint, per-event/class metrics, predictions, calibration outputs, and figures. The report combines only completed saved test artifacts; it does not infer missing results.